# llminfer README Companion (Colab)

This notebook is a concise, runnable companion to the README.

Notebook map:
- **Quickstart:** `llminfer_colab.ipynb`
- **Advanced tuning:** `llminfer_advanced_colab.ipynb`
- **Serving API:** `llminfer_serving_colab.ipynb`
- **Everything together:** `llminfer_all_examples_colab.ipynb`



## 0) Runtime
In Colab, choose **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
!nvidia-smi

import os
import platform
import torch

print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not enabled. Use a GPU runtime for realistic results.')


## 1) Clone + install

In [ ]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone -q {REPO_URL} {TARGET_DIR}
%cd {TARGET_DIR}

!pip -q install -U pip
!pip -q install -e ".[serve,peft]"


## 2) Helpers

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

def header(title: str):
    print('\n' + '=' * 20 + f' {title} ' + '=' * 20)

def show_image(path: str):
    p = Path(path)
    if p.exists():
        print(f'[plot] {p}')
        display(Image(filename=str(p)))
    else:
        print(f'[missing plot] {p}')


## 3) Quick README flow: run + stream + batch

In [ ]:
from llminfer import InferenceEngine, EngineConfig

MODEL = 'facebook/opt-125m'
engine = InferenceEngine(EngineConfig(model_name=MODEL))
engine.load()

header('run()')
r = engine.run('Explain transformer attention in 3 bullets.', max_new_tokens=96, temperature=0.2)
print(r.generated_text)
print('Stats:', r.stats)

header('stream()')
for chunk in engine.stream('Explain KV cache briefly.', max_new_tokens=80, temperature=0.2):
    if not chunk.is_final:
        print(chunk.token, end='', flush=True)
    else:
        print('\nFinal:', chunk.stats)

header('run_batch()')
batch = engine.run_batch([
    'One line on CUDA graphs.',
    'One line on paged KV cache.',
], max_new_tokens=40, temperature=0.2)
for i, out in enumerate(batch, 1):
    print(f'{i}. {out.generated_text.strip()}')


## 4) Advanced decode controls (README section)

In [ ]:
res = engine.run(
    'Explain GPU memory hierarchy in 5 bullets.',
    max_new_tokens=160,
    temperature=0.2,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.05,
    no_repeat_ngram_size=3,
    stop_sequences=['\n\nReferences:'],
    bad_words=["I'm not sure", 'joking'],
    force_words=['HBM', 'Tensor Core'],
    seed=42,
)
header('Advanced decode output')
print(res.generated_text)


## 5) Benchmark and plot suite (README artifacts section)

In [ ]:
from llminfer import Benchmarker

bm = Benchmarker(engine)
bench = bm.run(batch_sizes=[1, 2, 4], num_runs=3, warmup_runs=1, max_new_tokens=48, use_continuous_batching=True)
bench.print_summary()

outdir = Path('benchmarks/readme')
outdir.mkdir(parents=True, exist_ok=True)
bench.save_json(str(outdir / 'benchmark.json'))
bench.save_csv(str(outdir / 'benchmark.csv'))
plots = bench.plot_suite(str(outdir), prefix='readme')
print('Artifacts in', outdir.resolve())
print('Plot files:', plots)

for name in ['readme_dashboard.png', 'readme_throughput.png', 'readme_latency.png', 'readme_memory.png']:
    show_image(str(outdir / name))


## 6) Backend comparison (README compare section)

In [ ]:
from llminfer.benchmark import BackendComparison
from llminfer.config import Backend

cmp = BackendComparison(
    model_name='facebook/opt-125m',
    backends=[Backend.EAGER, Backend.COMPILED],
    compile_cudagraphs=True,
)
cmp_results = cmp.run(batch_sizes=[1, 2, 4], num_runs=2, max_new_tokens=40, use_continuous_batching=True)
cmp.print_table(cmp_results)

cmp_dir = Path('benchmarks/readme_compare')
cmp_dir.mkdir(parents=True, exist_ok=True)
cmp.plot(cmp_results, str(cmp_dir / 'comparison.png'))
show_image(str(cmp_dir / 'comparison.png'))


## 7) Cleanup

In [ ]:
try:
    engine.unload()
except Exception:
    pass
print('Done.')
